In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
#from pypdf import PdfReader
import gradio as gr
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [5]:
generator = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
evaluator = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

In [8]:
class eval(BaseModel):
    is_acceptable: bool = Field(description="True if response is professional else false")
    feedback: str = Field(description="Feedback to make respond professional", default="")

In [27]:
sys_gen = """
You are a question answer agent, you answer user question in a professional way. Only respond with the user answer only.
"""

sys_eval = """
You are an evaluator agent, you evaluate a response from a QA agent. 
Use true to indicate response is professional and false to indicate response is not professional.
Also give feedback on how to make response professional if response is not acceptable.

## Answer: {}

Respond with json only.
example:
{{
"is_acceptable": "False",
"feedback": ""
}}
"""

In [28]:
def chat(message, history):
    messages = [{"role": "system", "content": sys_gen}] + [{"role": h["role"], "content": h["content"]} for h in history] + [{"role": "user", "content": message}]

    answer = generator.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
    answer = answer.choices[0].message.content
    print(answer)

    eval_messages = [{"role": "system", "content": sys_eval.format(answer)}]
    eval_result = evaluator.beta.chat.completions.parse(model="qwen/qwen3.5-9b", messages=eval_messages, response_format=eval)
    eval_result = eval_result.choices[0].message.parsed

    if not eval_result.is_acceptable:
        print("not acceptable")
        print(eval_result.feedback)
        updated_sys_gen = sys_gen + f"\n\n You responded with {answer} but the quality control system rejected it with this feedback {eval_result.feedback}"
        updated_messages = [{"role": "system", "content": updated_sys_gen}] + [{"role": h["role"], "content": h["content"]} for h in history] + [{"role": "user", "content": message}]
        answer = generator.chat.completions.create(model="llama-3.1-8b-instant", messages=updated_messages)

    return answer.choices[0].message.content

    

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Atay errmat nay in igpay atin.


Traceback (most recent call last):
  File "/home/elijah/elijah/ai-bc/projects/my_projects_agentic/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/elijah/elijah/ai-bc/projects/my_projects_agentic/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/elijah/elijah/ai-bc/projects/my_projects_agentic/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/elijah/elijah/ai-bc/projects/my_projects_agentic/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/hom

In Iay, atttery is aay owhay ameay.
not acceptable
The response is unintelligible and appears to be gibberish or corrupted text. It contains severe spelling errors and does not convey any meaningful information. To improve professional quality, the response should be coherent, grammatically correct, and provide relevant information to the user.
